# Day 13 — Transformers & BERT — Text Intelligence
## RetailMind AI

**Goal:** Fine-tune a pretrained BERT model for Amazon review sentiment, create a small support-ticket routing experiment, compare against a TF-IDF baseline, and save reusable model artifacts.

> Run this notebook from the `RETAILMIND_AI` repository root. The notebook uses Hugging Face Transformers for model/tokenizer/training, PyTorch as the required model backend, Hugging Face Datasets for loading `amazon_polarity`, and scikit-learn for evaluation/baselines.

## 1. Install dependencies

Run this cell once in the active project environment. If `torch` installation fails with `getaddrinfo failed`, fix the internet/DNS connection first; that error is not a BERT code error.

In [6]:
%pip install -U transformers datasets accelerate torch scikit-learn pandas numpy joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and reproducibility

In [7]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    pipeline,
)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "models" / "day13_bert"
REPORT_DIR = ROOT / "reports"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Project root:", ROOT)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

Project root: c:\Users\ANSHU SHAW\OneDrive\Desktop\RETAILMIND_AI\notebooks
Device: cpu
PyTorch: 2.15.0+cpu


## 3. Load Amazon Polarity

Amazon Polarity contains binary sentiment labels: `0 = negative`, `1 = positive`. To keep the first local run manageable, the notebook samples a subset. Increase the sample sizes after confirming that the full workflow runs on your machine.

In [8]:
# First run downloads the dataset from Hugging Face; internet access is required.
raw = load_dataset("fancyzhx/amazon_polarity")

TRAIN_LIMIT = 2000
TEST_LIMIT = 500

train_df = pd.DataFrame(raw["train"].shuffle(seed=SEED).select(range(TRAIN_LIMIT)))
test_df = pd.DataFrame(raw["test"].shuffle(seed=SEED).select(range(TEST_LIMIT)))

# Combine title and content into one input string.
def combine_review(row):
    title = str(row.get("title", "") or "")
    content = str(row.get("content", "") or "")
    return (title + ". " + content).strip(". ").strip()

train_df["text"] = train_df.apply(combine_review, axis=1)
test_df["text"] = test_df.apply(combine_review, axis=1)
train_df = train_df[["text", "label"]].dropna()
test_df = test_df[["text", "label"]].dropna()

print(train_df.shape, test_df.shape)
display(train_df.head())
print(train_df["label"].value_counts())

(2000, 2) (500, 2)


,text,label
0,Anyone who likes this better than the Pekinpah...,0
1,Author seems mentally unstable. I know that To...,0
2,Spaetzle Noodles. This type of spaetzle maker ...,1
3,Cool product until the cats lost it . . . .. M...,1
4,Works as expected. I've had no problems runnin...,1


label
0    1011
1     989
Name: count, dtype: int64


## 4. Day-12-style TF-IDF baseline

This baseline gives a fair reference point. If your Day 12 notebook already saved a stronger classical model, load and evaluate that model instead.

In [9]:
X_train, y_train = train_df["text"], train_df["label"]
X_test, y_test = test_df["text"], test_df["label"]

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), stop_words="english")
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

baseline = LogisticRegression(max_iter=1000, n_jobs=-1, random_state=SEED)
baseline.fit(X_train_tfidf, y_train)
baseline_pred = baseline.predict(X_test_tfidf)

def metric_row(name, y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_weighted": p,
        "recall_weighted": r,
        "f1_weighted": f1,
    }

baseline_metrics = metric_row("TF-IDF + LogisticRegression", y_test, baseline_pred)
pd.DataFrame([baseline_metrics])

c:\Users\ANSHU SHAW\OneDrive\Desktop\RETAILMIND_AI\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,model,accuracy,precision_weighted,recall_weighted,f1_weighted
0,TF-IDF + LogisticRegression,0.816,0.816116,0.816,0.816006


## 5. Tokenize and prepare BERT

`bert-base-uncased` is a general English BERT checkpoint. The tokenizer converts text into `input_ids` and `attention_mask`. Truncation caps long reviews at 256 tokens for this experiment.

In [10]:
CHECKPOINT = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
    )

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

train_tokenized = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
test_tokenized = test_ds.map(tokenize_batch, batched=True, remove_columns=["text"])

train_tokenized = train_tokenized.rename_column("label", "labels")
test_tokenized = test_tokenized.rename_column("label", "labels")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(train_tokenized[0].keys())

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])


## 6. Fine-tune BERT for sentiment

For a first run, use a small subset and one epoch. Fine-tuning BERT is memory-intensive; reduce `TRAIN_LIMIT`, `TEST_LIMIT`, and batch size if your computer runs out of RAM/VRAM. CPU training may be slow.

In [11]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "sentiment_checkpoints"),
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision_weighted": p,
        "recall_weighted": r,
        "f1_weighted": f1,
    }

sentiment_trainer = Trainer(
    model=sentiment_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

sentiment_trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\ANSHU SHAW\On

Epoch,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted
1,0.385657,0.278611,0.922000,0.922859,0.922000,0.921983


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=500, training_loss=0.37486518478393555, metrics={'train_runtime': 2576.48, 'train_samples_per_second': 0.776, 'train_steps_per_second': 0.194, 'total_flos': 166430075845920.0, 'train_loss': 0.37486518478393555, 'epoch': 1.0})

## 7. Evaluate BERT and compare with TF-IDF

In [12]:
bert_eval = sentiment_trainer.evaluate()
bert_pred_output = sentiment_trainer.predict(test_tokenized)
bert_pred = np.argmax(bert_pred_output.predictions, axis=-1)

bert_metrics = metric_row("BERT fine-tuned", y_test, bert_pred)
comparison = pd.DataFrame([baseline_metrics, bert_metrics])
display(comparison)
print("BERT classification report:")
print(classification_report(y_test, bert_pred, target_names=["negative", "positive"], zero_division=0))

comparison.to_csv(REPORT_DIR / "day13_sentiment_comparison.csv", index=False)
with open(REPORT_DIR / "day13_sentiment_metrics.json", "w", encoding="utf-8") as f:
    json.dump({"baseline": baseline_metrics, "bert": bert_metrics}, f, indent=2)

c:\Users\ANSHU SHAW\OneDrive\Desktop\RETAILMIND_AI\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted
0.385657,0.278611,1,0.922000,0.922859,0.922000,0.921983


c:\Users\ANSHU SHAW\OneDrive\Desktop\RETAILMIND_AI\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


,model,accuracy,precision_weighted,recall_weighted,f1_weighted
0,TF-IDF + LogisticRegression,0.816,0.816116,0.816,0.816006
1,BERT fine-tuned,0.922,0.922859,0.922,0.921983


BERT classification report:
              precision    recall  f1-score   support

    negative       0.94      0.90      0.92       253
    positive       0.90      0.94      0.92       247

    accuracy                           0.92       500
   macro avg       0.92      0.92      0.92       500
weighted avg       0.92      0.92      0.92       500



## 8. Save the BERT sentiment model

The saved directory contains the model weights and tokenizer files. Keep the model directory in the repository only if its size is acceptable; otherwise store it outside Git and document how to recreate it.

In [13]:
SENTIMENT_DIR = MODEL_DIR / "sentiment"
sentiment_trainer.save_model(str(SENTIMENT_DIR))
tokenizer.save_pretrained(str(SENTIMENT_DIR))
print("Saved to:", SENTIMENT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: c:\Users\ANSHU SHAW\OneDrive\Desktop\RETAILMIND_AI\notebooks\models\day13_bert\sentiment


## 9. Create support-ticket labels from review text

This is a **proxy experiment**, not a real support-ticket dataset. We assign categories using transparent keyword rules, then train a second BERT classifier on the same review text. Replace this section with genuine, manually reviewed support tickets when available.

Categories:
- `delivery`: shipping, late delivery, arrival
- `product_quality`: broken, damaged, defective, quality
- `refund_payment`: refund, charge, payment, billing
- `other`: none of the above

In [14]:
import re

ROUTING_RULES = {
    "delivery": [
        "delivery", "shipping", "shipment", "arrived", "arrival",
        "late", "package", "tracking", "courier"
    ],
    "product_quality": [
        "broken", "damaged", "defective", "quality", "cracked",
        "missing part", "not working", "faulty"
    ],
    "refund_payment": [
        "refund", "money back", "charged", "charge", "payment",
        "billing", "invoice", "overcharged"
    ],
}

def assign_ticket_category(text):
    text_lower = str(text).lower()
    for category, keywords in ROUTING_RULES.items():
        if any(keyword in text_lower for keyword in keywords):
            return category
    return "other"

routing_df = train_df.copy()
routing_df["route_label"] = routing_df["text"].apply(assign_ticket_category)

print(routing_df["route_label"].value_counts())
display(routing_df[["text", "route_label"]].head(10))

route_label
other              1650
delivery            191
product_quality     124
refund_payment       35
Name: count, dtype: int64


,text,route_label
0,Anyone who likes this better than the Pekinpah...,other
1,Author seems mentally unstable. I know that To...,other
2,Spaetzle Noodles. This type of spaetzle maker ...,other
3,Cool product until the cats lost it . . . .. M...,other
4,Works as expected. I've had no problems runnin...,other
5,Not a bad deal.... If you're considering purch...,product_quality
6,Nice extras on the 300 DVD. Good picture on bl...,other
7,Too Mellow for Me. I've always been a John Mel...,other
8,just what I needed. I need t's like this to ge...,delivery
9,"nasty. In 1880s Australia, a lawman makes a de...",other


## 10. Train BERT for ticket routing

A stratified split preserves the category proportions. The keyword-derived labels may be noisy, so interpret these scores as a pipeline demonstration rather than proof of real ticket-routing quality.

In [15]:
route_labels = sorted(routing_df["route_label"].unique())
route_label2id = {label: i for i, label in enumerate(route_labels)}
route_id2label = {i: label for label, i in route_label2id.items()}

route_train, route_test = train_test_split(
    routing_df[["text", "route_label"]],
    test_size=0.2,
    random_state=SEED,
    stratify=routing_df["route_label"],
)

route_train = route_train.rename(columns={"route_label": "label"}).copy()
route_test = route_test.rename(columns={"route_label": "label"}).copy()
route_train["label"] = route_train["label"].map(route_label2id)
route_test["label"] = route_test["label"].map(route_label2id)

route_train_ds = Dataset.from_pandas(route_train.reset_index(drop=True))
route_test_ds = Dataset.from_pandas(route_test.reset_index(drop=True))

route_train_tok = route_train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
route_test_tok = route_test_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
route_train_tok = route_train_tok.rename_column("label", "labels")
route_test_tok = route_test_tok.rename_column("label", "labels")

route_model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=len(route_labels),
    id2label=route_id2label,
    label2id=route_label2id,
)

route_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "routing_checkpoints"),
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

route_trainer = Trainer(
    model=route_model,
    args=route_args,
    train_dataset=route_train_tok,
    eval_dataset=route_test_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

route_trainer.train()

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\ANSHU SHAW\On

Epoch,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted
1,0.664306,0.710464,0.825000,0.680625,0.825000,0.745890


TrainOutput(global_step=400, training_loss=0.6844024085998535, metrics={'train_runtime': 2452.4054, 'train_samples_per_second': 0.652, 'train_steps_per_second': 0.163, 'total_flos': 129611244393696.0, 'train_loss': 0.6844024085998535, 'epoch': 1.0})

## 11. Evaluate routing per class

In [16]:
route_output = route_trainer.predict(route_test_tok)
route_pred_ids = np.argmax(route_output.predictions, axis=-1)
route_true_ids = np.array(route_test["label"])

print(classification_report(
    route_true_ids,
    route_pred_ids,
    labels=list(range(len(route_labels))),
    target_names=route_labels,
    zero_division=0,
))

route_metrics = metric_row("BERT ticket routing", route_true_ids, route_pred_ids)
with open(REPORT_DIR / "day13_ticket_routing_metrics.json", "w", encoding="utf-8") as f:
    json.dump(
        {"labels": route_labels, "metrics": route_metrics},
        f,
        indent=2,
    )

c:\Users\ANSHU SHAW\OneDrive\Desktop\RETAILMIND_AI\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


                 precision    recall  f1-score   support

       delivery       0.00      0.00      0.00        38
          other       0.82      1.00      0.90       330
product_quality       0.00      0.00      0.00        25
 refund_payment       0.00      0.00      0.00         7

       accuracy                           0.82       400
      macro avg       0.21      0.25      0.23       400
   weighted avg       0.68      0.82      0.75       400



## 12. Save the ticket-routing model and label map

In [17]:
ROUTING_DIR = MODEL_DIR / "routing"
route_trainer.save_model(str(ROUTING_DIR))
tokenizer.save_pretrained(str(ROUTING_DIR))

with open(ROUTING_DIR / "route_labels.json", "w", encoding="utf-8") as f:
    json.dump(
        {"label2id": route_label2id, "id2label": route_id2label},
        f,
        indent=2,
    )

print("Routing model saved to:", ROUTING_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Routing model saved to: c:\Users\ANSHU SHAW\OneDrive\Desktop\RETAILMIND_AI\notebooks\models\day13_bert\routing


## 13. Inference demo

In [18]:
sentiment_pipe = pipeline(
    "text-classification",
    model=str(SENTIMENT_DIR),
    tokenizer=str(SENTIMENT_DIR),
    device=0 if torch.cuda.is_available() else -1,
)

routing_pipe = pipeline(
    "text-classification",
    model=str(ROUTING_DIR),
    tokenizer=str(ROUTING_DIR),
    device=0 if torch.cuda.is_available() else -1,
)

sample_review = "The package arrived late and the box was damaged."
print("Sentiment:", sentiment_pipe(sample_review))
print("Ticket route:", routing_pipe(sample_review))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Sentiment: [{'label': 'NEGATIVE', 'score': 0.991754412651062}]
Ticket route: [{'label': 'other', 'score': 0.8951718807220459}]


## 14. Replace the classical sentiment module in RetailMind

Use the saved model directory from your existing prediction/service code. This helper returns a consistent dictionary so it can be integrated into a Streamlit app or API.

Do not delete the Day 12 model yet. Keep it as a fallback until the BERT model is evaluated on the same held-out dataset and shows a worthwhile improvement.

In [19]:
def predict_review_sentiment(text):
    result = sentiment_pipe(str(text), truncation=True, max_length=256)[0]
    return {
        "sentiment": result["label"].lower(),
        "confidence": float(result["score"]),
        "model": "bert-base-uncased-finetuned-day13",
    }

def route_support_ticket(text):
    result = routing_pipe(str(text), truncation=True, max_length=256)[0]
    return {
        "category": result["label"],
        "confidence": float(result["score"]),
        "model": "bert-ticket-router-day13",
    }

print(predict_review_sentiment("Excellent product, works perfectly."))
print(route_support_ticket("I was charged twice and need a refund."))

{'sentiment': 'positive', 'confidence': 0.9967368245124817, 'model': 'bert-base-uncased-finetuned-day13'}
{'category': 'other', 'confidence': 0.9004220962524414, 'model': 'bert-ticket-router-day13'}


## 15. Save a compact Day 13 summary

In [20]:
summary = {
    "day": 13,
    "topic": "Transformers and BERT",
    "sentiment_model": str(SENTIMENT_DIR.as_posix()),
    "routing_model": str(ROUTING_DIR.as_posix()),
    "baseline_metrics": baseline_metrics,
    "bert_sentiment_metrics": bert_metrics,
    "ticket_routing_metrics": route_metrics,
    "dataset": "fancyzhx/amazon_polarity",
    "train_samples": int(len(train_df)),
    "test_samples": int(len(test_df)),
    "device": DEVICE,
}
with open(REPORT_DIR / "day13_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

{
  "day": 13,
  "topic": "Transformers and BERT",
  "sentiment_model": "c:/Users/ANSHU SHAW/OneDrive/Desktop/RETAILMIND_AI/notebooks/models/day13_bert/sentiment",
  "routing_model": "c:/Users/ANSHU SHAW/OneDrive/Desktop/RETAILMIND_AI/notebooks/models/day13_bert/routing",
  "baseline_metrics": {
    "model": "TF-IDF + LogisticRegression",
    "accuracy": 0.816,
    "precision_weighted": 0.8161162258596137,
    "recall_weighted": 0.816,
    "f1_weighted": 0.8160058880942095
  },
  "bert_sentiment_metrics": {
    "model": "BERT fine-tuned",
    "accuracy": 0.922,
    "precision_weighted": 0.9228590556730092,
    "recall_weighted": 0.922,
    "f1_weighted": 0.9219828382838284
  },
  "ticket_routing_metrics": {
    "model": "BERT ticket routing",
    "accuracy": 0.825,
    "precision_weighted": 0.680625,
    "recall_weighted": 0.825,
    "f1_weighted": 0.7458904109589041
  },
  "dataset": "fancyzhx/amazon_polarity",
  "train_samples": 2000,
  "test_samples": 500,
  "device": "cpu"
}


## 16. Day 13 completion checklist

- [ ] Amazon Polarity loaded and prepared
- [ ] TF-IDF baseline trained and evaluated
- [ ] BERT sentiment model fine-tuned and evaluated
- [ ] Sentiment comparison saved
- [ ] Proxy ticket categories created and reviewed
- [ ] BERT ticket-routing model trained and evaluated per class
- [ ] Both models saved under `models/day13_bert/`
- [ ] Inference helpers tested
- [ ] Day 13 reports saved under `reports/`

### Important interpretation notes
1. Compare BERT and TF-IDF using the same test rows and labels.
2. BERT is not automatically better; consider F1, per-class recall, inference latency, memory use, and model size.
3. Ticket labels here are generated by keyword rules from product reviews. Real ticket data and human-verified labels are required before production use.
4. If training is too slow, lower sample counts and train for one epoch first. Do not claim a model lift unless the measured results support it.

## 17. Suggested Git upload

From the repository root, inspect the files first. Avoid committing large model checkpoint folders unless your Git/LFS setup supports them.

```powershell
git status
git add notebooks/Day_13_Transformers_BERT_Text_Intelligence.ipynb reports/day13_sentiment_comparison.csv reports/day13_sentiment_metrics.json reports/day13_ticket_routing_metrics.json reports/day13_summary.json
git commit -m "Day 13: add BERT text intelligence"
git push origin main
```

If your notebook has a different filename, update the `git add` path. If you decide to version model artifacts, use Git LFS or document the Hugging Face checkpoint and local save path.